# Test Notebook for Reconfiguration Agent

This notebook loads a trained `h_ppo` agent and runs it in the `RecfgEnv` environment to evaluate its performance.

In [ ]:
import os
import sys
import torch as T

import plotly
from IPython.display import display 
from IPython.core.display import HTML

## Tomas Mazak's workaround (Make latex in plotly work in Jupyter notebooks)
plotly.offline.init_notebook_mode()
display(HTML(
    '<script type="text/javascript" async src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.1/MathJax.js?config=TeX-MML-AM_SVG"></script>'
))
##

# Ensure the project root is in the path
script_dir = os.path.abspath('')
project_root = os.path.abspath(os.path.join(script_dir, '..', '..'))
sys.path.append(project_root)

from src.leo_gym.gyms.recfg_gym import RecfgEnv
from src.leo_gym.rl_algorithms.h_ppo.h_ppo_agent import Agent
from notebooks.C4_Reconfiguration.train_recfg_hppo_cfg import env_cfg, ppo_cfg

## 1. Set the Path to the Trained Model

You need to specify the path to the directory containing the saved model files (`policynet.pth` and `valuenet.pth`). These are saved by the training script inside the `mlruns` directory.

**Action Required:** Replace `<experiment_id>` and `<run_id>` with the actual IDs from your MLflow training run. You can also point it to the `final` saved model or a specific checkpoint.

In [ ]:
# TODO: Update this path to your trained model directory
MODEL_DIR = os.path.join(project_root, 'mlruns', '423578571969266556', 'ca83a685e39641a6a64ecaa2a093bc4f', 'artifacts', 'models', 'final')

policy_net_path = os.path.join(MODEL_DIR, 'policynet.pth')
value_net_path = os.path.join(MODEL_DIR, 'valuenet.pth')

if not os.path.exists(policy_net_path) or not os.path.exists(value_net_path):
    raise FileNotFoundError(f"Model files not found in {MODEL_DIR}. Please update the path.")

print(f"Loading models from: {MODEL_DIR}")

## 2. Initialize Environment and Agent

In [ ]:
# Initialize the environment
env = RecfgEnv(cfg=env_cfg, seed=101) # Use a different seed for testing

# Instantiate the Agent
# The agent needs the environment's observation and action spaces for initialization
agent = Agent(
    env_obs=env.observation_space,
    env_actions=env.action_space,
    ppo_cfg=ppo_cfg,
    env_cfg=env_cfg,
)

# Load the trained network weights
agent.load_trained_networks(
    train=False, # Set to False for evaluation
    device=T.device('cpu'),
    file_name_policy=policy_net_path,
    file_name_critic=value_net_path
)

print("Agent and environment initialized successfully.")

## 3. Run the Evaluation Loop

In [ ]:
try:
    obs, _ = env.reset()
    
    while True:
        # Use agent.choose_action for the custom agent
        # We use deterministic=True for evaluation to get the most likely action
        action_dis, action_cont, _, _, _ = agent.choose_action(obs, deterministic=True)
        action = {"discrete": action_dis, "continuous": action_cont}
        
        obs, rewards, terminated, truncated, info = env.step(action)
        
        if terminated or truncated:
            print("Episode finished.")
            break
                                                        
except KeyboardInterrupt:
    print("Evaluation interrupted.")

# The satellite state is stored in the environment
satellite = env.satellite

## 4. Plot the Results

In [ ]:
satellite.plot_states_interactive()